# Gemini Ask CIP | CPU Preflight First, Bounded T4 Training Only After ALLOW

**Keep this notebook private.** Never upload respondent-level data, secrets, or unapproved client surveys.

### Step 1 (Current Bundle): CPU Preflight Only — Zero GPU Units
1. Select **Runtime → Change runtime type → CPU**, **Python 3**, **Latest**, **High-RAM off**.
2. Select **Runtime → Run all** and upload `cip-colab-inputs.zip` (containing `Gemini-ask-cip-train.py` or `ask_cip_train.py`, `cases.jsonl`, `dataset-report.json`, and the 3 `ask-reader/` files).
3. The notebook validates inputs using pure Python stdlib **before** installing any ML dependencies or initializing GPU work. When blocked, it emits `Gemini-preflight-report.json` with `decision: "BLOCK"`, skips all later training cells, and downloads the report (with manual Files-panel fallback).

### Step 2 (Only After a Rebuilt, Independently Reviewed Bundle Passes Preflight)
1. Select **T4 GPU** with standard RAM (**High-RAM off**; L4 only if T4 is unavailable; do **not** default to A100).
2. Run **1 epoch** with explicit step/time caps (`--max-steps 2000 --max-minutes 30`).
3. Downloads `Gemini-cip-candidate.zip` stamped `NOT APPROVED FOR PROMOTION` until local isolated strict CIP replay passes. Keep `models/ask-reader` unchanged.

In [ ]:
# Cell 1: Fresh upload & ZIP integrity check (never silently reuses stale /content inputs)
from io import BytesIO
from pathlib import Path
import shutil
from zipfile import ZipFile

from google.colab import files

work = Path("/content/gemini-ask-run")
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True)

cases = work / "cases.jsonl"
provenance = work / "dataset-report.json"
baseline = work / "ask-reader"
output = work / "Gemini-cip-candidate"
model_files = ("model_optimized.onnx", "tokenizer.json", "calibration.json")

print("Upload cip-colab-inputs.zip...")
uploaded = files.upload()
archive = uploaded.get("cip-colab-inputs.zip")
if archive is None:
    raise ValueError("Expected a file named cip-colab-inputs.zip.")

with ZipFile(BytesIO(archive)) as bundle:
    bad = bundle.testzip()
    if bad:
        raise ValueError(f"Corrupt member in input ZIP: {bad}")
    names = set(bundle.namelist())
    script_name = "Gemini-ask-cip-train.py" if "Gemini-ask-cip-train.py" in names else "ask_cip_train.py"
    needed = [script_name, "cases.jsonl", *(f"ask-reader/{m}" for m in model_files)]
    missing = [n for n in needed if n not in names]
    if missing:
        raise ValueError("Missing required files in ZIP: " + ", ".join(missing))
    baseline.mkdir(exist_ok=True)
    for n in [*needed, "dataset-report.json"]:
        if n in names:
            dest = work / n
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(bundle.read(n))
script = work / script_name
del archive, uploaded
print(f"Extracted {script.name}, cases.jsonl, and shipped baseline into {work}.")

In [ ]:
# Cell 2: Pure-stdlib CPU preflight gate (runs BEFORE any pip install or GPU initialization)
import json
import subprocess
import sys

preflight_report = work / "Gemini-preflight-report.json"
check = subprocess.run(
    [sys.executable, "-S", str(script), "preflight", "--cases", str(cases),
     "--baseline", str(baseline), "--output", str(preflight_report)],
    check=False,
)
if check.returncode not in (0, 2) or not preflight_report.is_file():
    raise RuntimeError(f"Preflight failed unexpectedly (exit {check.returncode}).")
evidence = json.loads(preflight_report.read_text(encoding="utf-8"))
decision = evidence.get("decision", "ALLOW_NEXT_STEP" if evidence.get("ready") else "BLOCK")
ready = check.returncode == 0 and bool(evidence.get("ready")) and decision == "ALLOW_NEXT_STEP"
print(f"Preflight decision: {decision}")
if not ready:
    print("BLOCKED: Skipping ML installs and GPU training. Inspect Gemini-preflight-report.json.")
    for reason in evidence.get("blocking_reasons", []):
        print(" -", reason)

In [ ]:
# Cell 3: Bounded GPU training (strictly gated on preflight ALLOW_NEXT_STEP + CUDA GPU)
import os

trained = False
if not ready:
    print("Cell 3 skipped: preflight decision is BLOCK.")
else:
    import torch
    if not torch.cuda.is_available():
        print("Preflight passed (ALLOW_NEXT_STEP). Switch runtime to T4 GPU (standard RAM) to run training.")
    else:
        print("Assigned GPU:", torch.cuda.get_device_name(0))
        print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
        subprocess.run(
            [sys.executable, "-m", "pip", "-q", "install",
             "transformers==4.57.3", "onnx==1.20.0", "onnxruntime==1.23.2"],
            check=True,
        )
        train_cmd = [
            sys.executable, str(script), "train",
            "--cases", str(cases), "--baseline", str(baseline), "--output", str(output),
            "--epochs", "1", "--batch", "4", "--seed", "17",
        ]
        if script.name == "Gemini-ask-cip-train.py":
            train_cmd += ["--max-steps", "2000", "--max-minutes", "30"]
        subprocess.run(train_cmd, check=True, env={**os.environ, "HF_HUB_DISABLE_TELEMETRY": "1"})
        paired = output / "paired-report.json"
        if not paired.is_file() or json.loads(paired.read_text())["cases_sha256"] != evidence["cases_sha256"]:
            raise ValueError("Candidate paired-report.json missing or mismatched against cases_sha256.")
        trained = True

In [ ]:
# Cell 4: Package & download preflight report (if blocked/CPU) or Gemini-cip-candidate.zip (if trained)
from zipfile import ZIP_DEFLATED, ZipFile

delivery = preflight_report
if trained:
    paired = output / "paired-report.json"
    print(json.dumps(json.loads(paired.read_text(encoding="utf-8")), indent=2))
    delivery = work / "Gemini-cip-candidate.zip"
    members = [
        paired, preflight_report, *(output / m for m in model_files),
        output / "DEPLOY_AND_ROLLBACK.md", output / "SHA256SUMS.txt",
    ]
    with ZipFile(delivery, "w", compression=ZIP_DEFLATED) as bundle:
        for path in members:
            if path.is_file():
                bundle.write(path, f"{output.name}/{path.name}")
        if provenance.is_file():
            bundle.write(provenance, f"{output.name}/{provenance.name}")
    with ZipFile(delivery) as bundle:
        bad = bundle.testzip()
        if bad:
            raise ValueError(f"Candidate ZIP failed integrity check: {bad}")

print(f"Artifact ready at: {delivery}")
print("Attempting automatic browser download (if blocked, download manually from Colab Files panel)...")
try:
    files.download(str(delivery))
except Exception as exc:
    print(f"Automatic download unavailable ({exc}). Download manually from Files panel: {delivery}")

### After Download
1. Verify the downloaded file exists locally before disconnecting.
2. Select **Runtime → Disconnect and delete runtime** immediately to stop compute-unit billing.
3. Never replace `models/ask-reader` until isolated local strict replay (`scripts/ask_golden.py --strict`) passes all promotion gates.